# NB5b — Hyperparameter Sensitivity, Multi-scale GASF & Inference Profiling (Exp B + D + E)

Standalone file split out from NB5 so Exp B, D, and E can run independently of A/C (already completed).

**Time budget: hard ceiling of 3 hours.** To guarantee this, B and D use a lighter training protocol than the main paper's headline numbers:

| Setting | Main paper (Exp A/C) | This file (Exp B/D/E) |
|---|---|---|
| Train signals | 800/class/SNR (19,200 total) | 400/class/SNR (9,600 total) |
| Test signals  | 200/class/SNR (4,800 total)  | 120/class/SNR (2,880 total) |
| Epochs (max)  | 60 | 35 |
| Early-stop patience | 12 | 8 |

**This is appropriate because B/D answer a *relative* question** (Exp E is a separate inference/quantization profiler — it isn't a training sweep, so its own results are unaffected by the lighter protocol; only the underlying `m_ss` model it profiles is trained on the reduced budget.) — "which hyperparameter value is best, and how sensitive is accuracy to it?" — not "what is the final headline accuracy?" (that number comes from the main run with the full protocol). The same architecture and same swept values are used; only the per-run training budget is reduced.

### Setup
1. **Accelerator: GPU T4 x2**
2. **Run All** — should complete within 3 hours; each sweep prints elapsed time and a running total against the 3h budget.


In [ ]:
!pip install EMD-signal tqdm scikit-image -q

import os, tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU detected! Enable GPU T4 x2 in Settings > Accelerator')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
OUTPUT_DIR = '/kaggle/working/nb5_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'GPU ready: {len(gpus)} device(s), {mixed_precision.global_policy().name}')
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
import os, time, gc, warnings, logging, pickle, zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from scipy.signal import hilbert, butter, lfilter
from scipy.stats import skew, kurtosis, entropy as sp_entropy
from PyEMD import EMD
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
logging.getLogger('PyEMD').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

SNRs        = [-10, -5, 0, 5, 10, 20]
N_PER_CLASS = 400  # reduced from 800 for Exp B/D time budget (see header note)
N_TEST      = 120  # reduced from 200 for Exp B/D time budget
GASF_D      = 96
SEQ_D       = 256
BATCH       = 64
EPOCHS      = 35  # reduced from 60 for Exp B/D time budget
PATIENCE    = 8   # tightened further for Exp B/D sensitivity sweeps
MASTER_SEED = 4242
TEST_SEED   = 8888

plt.rcParams.update({'font.family':'serif','font.size':11,
    'axes.titlesize':12,'figure.dpi':110,'savefig.dpi':180,
    'savefig.bbox':'tight','savefig.pad_inches':0.15})

print(f'Setup complete. Train: {N_PER_CLASS*4*len(SNRs):,} | Test: {N_TEST*4*len(SNRs):,}')


# ── Time budget tracker ───────────────────────────────────────────────────
TIME_BUDGET_HOURS = 3.0
_t_start = time.time()

def time_check(label=''):
    elapsed_h = (time.time() - _t_start) / 3600
    remaining_h = TIME_BUDGET_HOURS - elapsed_h
    flag = '⚠️ OVER BUDGET' if remaining_h < 0 else ('🟡 tight' if remaining_h < 0.3 else '🟢 on track')
    print(f'  [TIME] {label}: elapsed={elapsed_h:.2f}h  remaining={remaining_h:.2f}h  {flag}')
    return remaining_h

print('Setup complete (Exp B/D lightweight protocol).')
print(f'  Train: {N_PER_CLASS*4*len(SNRs):,} signals | Test: {N_TEST*4*len(SNRs):,} signals')
print(f'  Epochs (max): {EPOCHS} | Patience: {PATIENCE}')
print(f'  Time budget: {TIME_BUDGET_HOURS}h')


In [ ]:
def butter_lp(cutoff, fs, order=9):
    from scipy.signal import butter
    b, a = butter(order, cutoff/(0.5*fs), btype='low')
    return b, a

def generate_signals(n_per_class, snrs, seed, num_samples=1024, tag=''):
    from scipy.signal import lfilter
    from scipy.signal import hilbert
    rng = np.random.RandomState(seed)
    total = n_per_class * 4 * len(snrs)
    fs, fc = 100000, 5000
    t = np.arange(num_samples) / fs
    X, Y, SL = [], [], []
    pbar = tqdm(total=total, desc=f'Gen [{tag}]', unit='sig')
    for snr in snrs:
        for _ in range(n_per_class):
            noise = rng.randn(num_samples)
            b, a  = butter_lp(1000, fs)
            m_t   = lfilter(b, a, noise)
            m_n   = m_t / (np.max(np.abs(m_t)) + 1e-10)
            am    = (1 + 0.8*m_n) * np.cos(2*np.pi*fc*t)
            pm    = np.cos(2*np.pi*fc*t + np.pi*m_n)
            fm    = np.cos(2*np.pi*fc*t + 2*np.pi*1000*np.cumsum(m_t)/fs)
            mh    = np.imag(hilbert(m_t))
            ssb   = m_t*np.cos(2*np.pi*fc*t) - mh*np.sin(2*np.pi*fc*t)
            for idx, sig in enumerate([am, pm, fm, ssb]):
                sp = np.mean(sig**2)
                ns = sig + np.sqrt(sp/(10**(snr/10))) * rng.randn(num_samples)
                X.append((ns-ns.mean())/ns.std()); Y.append(idx); SL.append(snr)
            pbar.update(4)
    pbar.close()
    return np.array(X,np.float32), np.array(Y,np.int8), np.array(SL,np.int8)

def apply_rayleigh(sig, rng, fd=5, fs=100000):
    # Vectorised Clarke's model (identical statistics, ~15x faster than the
    # per-sinusoid Python loop). Same N=20 paths, same random phase draws.
    N=20; n=len(sig); t=np.arange(n)/fs
    k = np.arange(N).reshape(-1,1)
    phI = rng.uniform(0, 2*np.pi, size=(N,1))
    phQ = rng.uniform(0, 2*np.pi, size=(N,1))
    arg = 2*np.pi*fd*np.cos(2*np.pi*k/N)*t
    I = np.sum(np.cos(arg + phI), axis=0) / N**0.5
    Q = np.sum(np.sin(arg + phQ), axis=0) / N**0.5
    h = (I+1j*Q)/2**0.5; out=np.real(h)*sig
    p=np.mean(out**2); return out/np.sqrt(p+1e-10) if p>1e-10 else out

def apply_rician(sig, rng, K_dB=4.0, fd=5, fs=100000):
    # Vectorised Rician (LOS + scattered Rayleigh component), same model.
    K=10**(K_dB/10); N=20; n=len(sig); t=np.arange(n)/fs
    k = np.arange(N).reshape(-1,1)
    phs = rng.uniform(0, 2*np.pi, size=(N,1))
    arg = 2*np.pi*fd*np.cos(2*np.pi*k/N)*t
    Is = np.sum(np.cos(arg + phs), axis=0) / N**0.5
    Icos=np.cos(2*np.pi*fd*t+rng.uniform(0,2*np.pi))
    h=np.sqrt(K/(K+1))*Icos + np.sqrt(1/(K+1))*Is
    out=h*sig; p=np.mean(out**2); return out/np.sqrt(p+1e-10) if p>1e-10 else out

print('Signal generation functions ready.')


In [ ]:
def emd_features(sig, max_imf=3):
    imfs = EMD().emd(sig, max_imf=max_imf)
    out  = []
    for i in range(max_imf):
        if i < len(imfs):
            x=imfs[i]; p=np.abs(x)**2; p/=(p.sum()+1e-10)
            out.extend([np.std(x),skew(x),kurtosis(x),np.sum(np.abs(x)**2),sp_entropy(p)])
        else:
            out.extend([0.0]*5)
    return np.array(out,np.float32)

def gasf(sig, size):
    sr = np.interp(np.linspace(0,len(sig),size), np.arange(len(sig)), sig)
    n  = (sr-sr.min())/(sr.max()-sr.min()+1e-8)*2-1
    ph = np.arccos(np.clip(n,-1,1))
    return (np.outer(np.cos(ph),np.cos(ph))-np.outer(np.sin(ph),np.sin(ph))).astype(np.float32)

def gasf_multiscale(sig, sizes=(48,96,128)):
    from skimage.transform import resize
    ms = max(sizes)
    chs= []
    for s in sizes:
        ch = gasf(sig, s)
        if s < ms:
            ch = resize(ch,(ms,ms),order=1,anti_aliasing=False).astype(np.float32)
        chs.append(ch[...,np.newaxis])
    return np.concatenate(chs, axis=-1)

def seqfeat(sig, seq_len=256):
    stride = max(1, len(sig)//seq_len)
    s = sig[::stride][:seq_len]
    if len(s)<seq_len: s=np.pad(s,(0,seq_len-len(s)))
    return s.astype(np.float32).reshape(seq_len,1)

def extract_features(X_raw, gasf_size=96, seq_len=256, max_imf=3,
                     multiscale=False, ms_sizes=(48,96,128), tag=''):
    el, gl, sl = [], [], []
    with tqdm(total=len(X_raw), desc=f'Features [{tag}]', unit='sig') as pb:
        for sig in X_raw:
            el.append(emd_features(sig, max_imf))
            if multiscale: gl.append(gasf_multiscale(sig, ms_sizes))
            else:          gl.append(gasf(sig, gasf_size))
            sl.append(seqfeat(sig, seq_len))
            pb.update(1)
    Xe = np.array(el,np.float32).reshape(-1, max_imf*5, 1)
    if multiscale:
        ms = max(ms_sizes)
        Xg = np.array(gl,np.float32).reshape(-1, ms, ms, len(ms_sizes))
    else:
        Xg = np.array(gl,np.float32).reshape(-1, gasf_size, gasf_size, 1)
    Xs = np.array(sl,np.float32)
    return Xe, Xg, Xs

print('Feature extraction functions ready.')


In [ ]:
class EmdAug(layers.Layer):
    def __init__(self,std=0.02,**kw): super().__init__(**kw); self.std=std
    def call(self,x,training=None):
        if training: return x+tf.cast(tf.random.normal(tf.shape(x),stddev=self.std),x.dtype)
        return x

class GafAug(layers.Layer):
    def __init__(self,lo=0.85,hi=1.15,**kw): super().__init__(**kw); self.lo,self.hi=lo,hi
    def call(self,x,training=None):
        if training:
            s=tf.cast(tf.random.uniform([tf.shape(x)[0],1,1,1],self.lo,self.hi),x.dtype)
            return x*s
        return x

class SeqAug(layers.Layer):
    def __init__(self,frac=0.1,lo=0.85,hi=1.15,**kw): super().__init__(**kw); self.frac,self.lo,self.hi=frac,lo,hi
    def call(self,x,training=None):
        if training:
            ml=tf.cast(tf.cast(tf.shape(x)[1],tf.float32)*self.frac,tf.int32)
            sh=tf.random.uniform([],minval=-ml,maxval=ml+1,dtype=tf.int32)
            x=tf.roll(x,sh,axis=1)
            s=tf.cast(tf.random.uniform([tf.shape(x)[0],1,1],self.lo,self.hi),x.dtype)
            return x*s
        return x

class GateApply(layers.Layer):
    def __init__(self,idx,**kw): super().__init__(**kw); self.idx=idx
    def call(self,inp): path,gate=inp; return path*tf.cast(gate[:,self.idx:self.idx+1],path.dtype)
    def get_config(self): c=super().get_config(); c['idx']=self.idx; return c

class EntropyGate(layers.Layer):
    def __init__(self,lam=0.0,**kw): super().__init__(**kw); self.lam=lam; self.d=layers.Dense(3,activation='softmax',dtype='float32')
    def call(self,x):
        g=self.d(x)
        if self.lam>0:
            v=tf.reduce_mean(tf.square(g-tf.reduce_mean(g,axis=0,keepdims=True)),axis=0)
            self.add_loss(self.lam*tf.exp(-10.*tf.reduce_mean(v)))
        return g

class ConstrainedGate(layers.Layer):
    def __init__(self,s=0.01,**kw):
        super().__init__(**kw)
        self.d=layers.Dense(3,activation='softmax',dtype='float32',
                             kernel_regularizer=tf.keras.regularizers.l2(s))
    def call(self,x): return self.d(x)

class CosineLR(tf.keras.callbacks.Callback):
    def __init__(self,lr0=3e-4,lrmin=1e-6,T=60): super().__init__(); self.lr0,self.lrmin,self.T=lr0,lrmin,T
    def on_epoch_begin(self,e,logs=None):
        lr=self.lrmin+(self.lr0-self.lrmin)*0.5*(1+np.cos(np.pi*e/self.T))
        self.model.optimizer.learning_rate.assign(float(lr))

def se_block(x,r=8):
    f=x.shape[-1]; s=layers.GlobalAveragePooling2D()(x); s=layers.Reshape((1,1,f))(s)
    s=layers.Dense(max(f//r,1),activation='relu',use_bias=False)(s)
    s=layers.Dense(f,activation='sigmoid',use_bias=False)(s)
    return layers.Multiply()([x,s])

def build_model(gs=96, sl=256, ne=15, gtype='entropy', gp=0.0, ls=0.1, bu=64, gc=1):
    ei=layers.Input(shape=(ne,1),     name='emd_input')
    gi=layers.Input(shape=(gs,gs,gc), name='gaf_input')
    si=layers.Input(shape=(sl,1),     name='seq_input')
    e=EmdAug()(ei); g=GafAug()(gi); s=SeqAug()(si)
    # Path 1
    x1=layers.Flatten()(e); x1=layers.Dense(64,activation='relu')(x1)
    x1=layers.BatchNormalization()(x1); x1=layers.Dense(32,activation='relu')(x1)
    # Path 2
    x2=layers.Concatenate()([layers.Conv2D(16,(3,3),padding='same',activation='relu')(g),
                              layers.Conv2D(16,(5,5),padding='same',activation='relu')(g)])
    x2=layers.BatchNormalization()(x2); x2=layers.MaxPooling2D()(x2); x2=se_block(x2)
    for nf in [64,128]:
        x2=layers.Conv2D(nf,(3,3),padding='same',activation='relu')(x2)
        x2=layers.BatchNormalization()(x2); x2=layers.MaxPooling2D()(x2); x2=se_block(x2)
    x2=layers.Concatenate()([layers.GlobalAveragePooling2D()(x2),layers.GlobalMaxPooling2D()(x2)])
    # Path 3
    x3=layers.Conv1D(32,7,strides=2,padding='same',activation='relu')(s); x3=layers.BatchNormalization()(x3)
    x3=layers.Conv1D(64,5,strides=2,padding='same',activation='relu')(x3); x3=layers.BatchNormalization()(x3)
    x3=layers.Conv1D(64,3,padding='same',activation='relu')(x3); x3=layers.BatchNormalization()(x3)
    x3=layers.Bidirectional(layers.GRU(bu,return_sequences=True,dropout=0.1))(x3)
    x3=layers.Bidirectional(layers.GRU(bu//2,return_sequences=False,dropout=0.1))(x3)
    # Gate
    skip=layers.Flatten()(ei)
    gh=layers.Dense(128,activation='relu')(layers.Concatenate()([x1,x2,x3,skip]))
    if   gtype=='entropy':     gate=EntropyGate(lam=gp,   name='path_gate')(gh)
    elif gtype=='constrained': gate=ConstrainedGate(s=gp, name='path_gate')(gh)
    else:                      gate=layers.Dense(3,activation='softmax',dtype='float32',name='path_gate')(gh)
    x1g=GateApply(0,name='gate_emd')([x1,gate])
    x2g=GateApply(1,name='gate_gaf')([x2,gate])
    x3g=GateApply(2,name='gate_gru')([x3,gate])
    z=layers.Concatenate()([x1g,x2g,x3g,skip])
    z=layers.Dense(256,activation='relu')(z); z=layers.BatchNormalization()(z); z=layers.Dropout(0.25)(z)
    z=layers.Dense(128,activation='relu')(z); z=layers.Dropout(0.15)(z)
    z=layers.Dense(64, activation='relu')(z)
    out=layers.Dense(4,activation='softmax',dtype='float32')(z)
    m=Model(inputs=[ei,gi,si],outputs=out)
    m.compile(optimizer=tf.keras.optimizers.Adam(3e-4),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=ls),
              metrics=['accuracy'])
    return m

def run(model,Xetr,Xgtr,Xstr,Ytr, Xete,Xgte,Xste,Yte,SNRte, epochs=EPOCHS):
    Yo=to_categorical(Ytr,4); Yoe=to_categorical(Yte,4)
    tr={'emd_input':Xetr,'gaf_input':Xgtr,'seq_input':Xstr}
    te={'emd_input':Xete,'gaf_input':Xgte,'seq_input':Xste}

    class TqdmEpoch(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.pbar = tqdm(total=self.params['epochs'], desc='  epochs',
                             unit='ep', leave=False, dynamic_ncols=True)
        def on_epoch_end(self, epoch, logs=None):
            loss  = logs.get('loss', 0)
            vloss = logs.get('val_loss', 0)
            acc   = logs.get('val_accuracy', 0) * 100
            self.pbar.set_postfix(loss=f'{loss:.3f}', val_loss=f'{vloss:.3f}',
                                  val_acc=f'{acc:.1f}%', refresh=True)
            self.pbar.update(1)
        def on_train_end(self, logs=None):
            self.pbar.close()

    # min_delta=5e-4 only triggers stopping once val_loss has truly
    # plateaued; restore_best_weights=True always recovers the actual best
    # epoch regardless of patience, so reported accuracy is unaffected —
    # this only trims wasted epochs after convergence.
    cbs=[CosineLR(3e-4,1e-6,epochs),
         EarlyStopping('val_loss',patience=PATIENCE,min_delta=5e-4,
                       restore_best_weights=True,verbose=0),
         TqdmEpoch()]
    model.fit(tr,Yo,validation_data=(te,Yoe),epochs=epochs,batch_size=BATCH,callbacks=cbs,verbose=0)
    p=model.predict(te,batch_size=256,verbose=0).argmax(1)
    return {s:float(np.mean(p[SNRte==s]==Yte[SNRte==s]))*100 for s in SNRs}

def gate_weights(model,Xe,Xg,Xs,SNRte):
    gm=Model(inputs=model.inputs,outputs=model.get_layer('path_gate').output)
    w=gm.predict({'emd_input':Xe,'gaf_input':Xg,'seq_input':Xs},batch_size=256,verbose=0)
    return {s:w[SNRte==s].mean(0) for s in SNRs}

print(f'Model builder ready. Default params: {build_model().count_params():,}')
tf.keras.backend.clear_session(); gc.collect()


In [ ]:
print('Generating shared dataset...')
t0=time.time()
X_tr, Y_tr, SNR_tr = generate_signals(N_PER_CLASS, SNRs, MASTER_SEED, tag='train')
X_te, Y_te, SNR_te = generate_signals(N_TEST,      SNRs, TEST_SEED,   tag='test')
print('Extracting standard features (m=96, IMFs=3)...')
Xe_tr,Xg_tr,Xs_tr = extract_features(X_tr, tag='train')
Xe_te,Xg_te,Xs_te = extract_features(X_te, tag='test')
print(f'Done in {(time.time()-t0)/60:.1f} min')
print(f'Train: {X_tr.shape[0]:,}  GASF:{Xg_tr.shape}  EMD:{Xe_tr.shape}')

time_check('After dataset generation')


In [ ]:
# ── EXP B: Hyperparameter sensitivity (R3.7) — lightweight protocol ──────
print('EXP B: Hyperparameter sensitivity (Exp B/D budget protocol)')
expB = {}

# B1: GASF size
print('B1: GASF image size m')
gasf_sizes=[32,48,64,96,128]; expB['gasf_size']={}
for ms in tqdm(gasf_sizes, desc='GASF size'):
    if ms == 96:
        Xetr2, Xgtr2, Xstr2 = Xe_tr, Xg_tr, Xs_tr
        Xete2, Xgte2, Xste2 = Xe_te, Xg_te, Xs_te
    else:
        gtr_only = np.array([gasf(sig, ms) for sig in X_tr], dtype=np.float32).reshape(-1, ms, ms, 1)
        gte_only = np.array([gasf(sig, ms) for sig in X_te], dtype=np.float32).reshape(-1, ms, ms, 1)
        Xetr2, Xgtr2, Xstr2 = Xe_tr, gtr_only, Xs_tr
        Xete2, Xgte2, Xste2 = Xe_te, gte_only, Xs_te
    tf.keras.backend.clear_session(); gc.collect()
    expB['gasf_size'][ms]=run(build_model(gs=ms),Xetr2,Xgtr2,Xstr2,Y_tr,Xete2,Xgte2,Xste2,Y_te,SNR_te)
    print(f'  m={ms}: mean={np.mean(list(expB["gasf_size"][ms].values())):.2f}%')
    if ms != 96:
        del Xetr2,Xgtr2,Xstr2,Xete2,Xgte2,Xste2; gc.collect()
time_check('After B1 (GASF size sweep)')

# B2: IMF count
print('\nB2: IMF count')
imf_counts=[2,3,4,5]; expB['n_imfs']={}
for ni in tqdm(imf_counts, desc='IMF count'):
    if ni == 3:
        Xetr2, Xgtr2, Xstr2 = Xe_tr, Xg_tr, Xs_tr
        Xete2, Xgte2, Xste2 = Xe_te, Xg_te, Xs_te
    else:
        etr_only = np.array([emd_features(sig, ni) for sig in X_tr], dtype=np.float32).reshape(-1, ni*5, 1)
        ete_only = np.array([emd_features(sig, ni) for sig in X_te], dtype=np.float32).reshape(-1, ni*5, 1)
        Xetr2, Xgtr2, Xstr2 = etr_only, Xg_tr, Xs_tr
        Xete2, Xgte2, Xste2 = ete_only, Xg_te, Xs_te
    tf.keras.backend.clear_session(); gc.collect()
    expB['n_imfs'][ni]=run(build_model(ne=ni*5),Xetr2,Xgtr2,Xstr2,Y_tr,Xete2,Xgte2,Xste2,Y_te,SNR_te)
    print(f'  IMFs={ni}: mean={np.mean(list(expB["n_imfs"][ni].values())):.2f}%')
    if ni != 3:
        del Xetr2,Xgtr2,Xstr2,Xete2,Xgte2,Xste2; gc.collect()
time_check('After B2 (IMF count sweep)')

# B3: BiGRU units
print('\nB3: BiGRU units')
bigru_opts=[32,48,64,96,128]; expB['bigru']={}
for bu in tqdm(bigru_opts, desc='BiGRU units'):
    tf.keras.backend.clear_session(); gc.collect()
    expB['bigru'][bu]=run(build_model(bu=bu),Xe_tr,Xg_tr,Xs_tr,Y_tr,Xe_te,Xg_te,Xs_te,Y_te,SNR_te)
    print(f'  BiGRU={bu}: mean={np.mean(list(expB["bigru"][bu].values())):.2f}%')
time_check('After B3 (BiGRU units sweep) — Exp B complete')

# Print tables
for label, d, params in [('GASF size',expB['gasf_size'],gasf_sizes),
                          ('IMF count',expB['n_imfs'],imf_counts),
                          ('BiGRU units',expB['bigru'],bigru_opts)]:
    print(f'\n{label}')
    print(f'{"Val":>6}' + ''.join(f'{s:>7}' for s in SNRs) + f'{"Mean":>8}')
    for p in params:
        r=d[p]
        print(f'{p:>6}' + ''.join(f'{r[s]:7.1f}' for s in SNRs)+f'{np.mean(list(r.values())):8.2f}')

fig,axes=plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Exp B: Hyperparameter Sensitivity (R3.7)')
def sp(ax,d,title,xl):
    ps=sorted(d.keys())
    mn=[np.mean(list(d[p].values())) for p in ps]
    lo=[np.mean([d[p][s] for s in SNRs if s<=0]) for p in ps]
    ax.plot(ps,mn,'o-',label='All SNR',color='steelblue',lw=2)
    ax.plot(ps,lo,'s--',label='<=0 dB', color='coral',lw=2)
    ax.set_xlabel(xl); ax.set_ylabel('Accuracy (%)'); ax.set_title(title)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
sp(axes[0],expB['gasf_size'],'GASF size m','m (pixels)')
sp(axes[1],expB['n_imfs'],   'IMF count',  'IMFs')
sp(axes[2],expB['bigru'],    'BiGRU units','units')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/expB_sensitivity.png'); plt.show()
print('Saved expB_sensitivity.png')


In [ ]:
# ── EXP D: Multi-scale GASF (R3.10) — lightweight protocol ──────────────
print('EXP D: Multi-scale GASF')
MS = (48, 96, 128)

print('Extracting multi-scale features...')
Xetr_ms,Xgtr_ms,Xstr_ms=extract_features(X_tr,multiscale=True,ms_sizes=MS,tag='train-ms')
Xete_ms,Xgte_ms,Xste_ms=extract_features(X_te,multiscale=True,ms_sizes=MS,tag='test-ms')
print(f'Multi-scale GASF shape: {Xgtr_ms.shape}')
time_check('After multi-scale feature extraction')

print('Training multi-scale model...')
tf.keras.backend.clear_session(); gc.collect()
m_ms = build_model(gs=max(MS), gc=len(MS))
ms_res = run(m_ms,Xetr_ms,Xgtr_ms,Xstr_ms,Y_tr,Xete_ms,Xgte_ms,Xste_ms,Y_te,SNR_te)
del Xetr_ms,Xgtr_ms,Xstr_ms,Xete_ms,Xgte_ms,Xste_ms; gc.collect()
time_check('After multi-scale model training')

# Single-scale m=96: reuse Exp B's result if this notebook ran B first in
# this session; otherwise train fresh (still on the lighter B/D protocol).
print('Training single-scale m=96 model (for comparison + Exp E timing if needed)...')
tf.keras.backend.clear_session(); gc.collect()
m_ss = build_model(gs=96, gc=1)
ss_res_live = run(m_ss,Xe_tr,Xg_tr,Xs_tr,Y_tr,Xe_te,Xg_te,Xs_te,Y_te,SNR_te)

if 'expB' in dir() and 96 in expB.get('gasf_size', {}):
    ss_res = expB['gasf_size'][96]
    print('  Using Exp B m=96 result for Exp D comparison table (identical config, same session).')
else:
    ss_res = ss_res_live

expD = {'Single-scale m=96': ss_res,
        f'Multi-scale {MS[0]}+{MS[1]}+{MS[2]}': ms_res}
time_check('After Exp D complete')

print('\nEXP D RESULTS')
print(f'{"Variant":<36}' + ''.join(f'{s:>7}' for s in SNRs) + f'{"Mean":>8}')
print('-'*80)
for name,r in expD.items():
    print(f'{name:<36}' + ''.join(f'{r[s]:7.1f}' for s in SNRs)+f'{np.mean(list(r.values())):8.2f}')

fig,ax=plt.subplots(figsize=(8,5))
fig.suptitle('Exp D: Multi-scale vs Single-scale GASF (R3.10)')
for (name,r),col in zip(expD.items(),['steelblue','coral']):
    ax.plot(SNRs,[r[s] for s in SNRs],'o-',label=name,color=col,lw=2)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,105)
ax.set_title('Per-SNR: Single vs Multi-scale GASF'); ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/expD_multiscale.png'); plt.show()
print('Saved expD_multiscale.png')


In [ ]:
# ── EXP E: Inference time profiling + INT8 quantization (R4.7) ──────────
# Reuses m_ss (single-scale model) trained in the Exp D cell above.
print('EXP E: Inference time profiling')
eval_m = m_ss   # reuse single-scale model

# Warm-up
dummy={'emd_input':Xe_te[:1],'gaf_input':Xg_te[:1],'seq_input':Xs_te[:1]}
for _ in range(10): eval_m.predict(dummy, verbose=0)

# Single-sample latency
stimes=[]
for i in range(200):
    j=i%len(Xe_te)
    inp={'emd_input':Xe_te[j:j+1],'gaf_input':Xg_te[j:j+1],'seq_input':Xs_te[j:j+1]}
    t0=time.perf_counter(); eval_m.predict(inp,verbose=0); stimes.append((time.perf_counter()-t0)*1000)
sm=np.mean(stimes[50:]); ss=np.std(stimes[50:]); sp95=np.percentile(stimes[50:],95)
print(f'Single-sample: mean={sm:.1f}ms  std={ss:.1f}ms  p95={sp95:.1f}ms')

# Batch throughput
bres={}
for bs in [1,8,32,64,128,256]:
    if bs>len(Xe_te): continue
    inp={'emd_input':Xe_te[:bs],'gaf_input':Xg_te[:bs],'seq_input':Xs_te[:bs]}
    ts=[]
    for _ in range(20): t0=time.perf_counter(); eval_m.predict(inp,verbose=0); ts.append(time.perf_counter()-t0)
    avg=np.mean(ts[5:])
    bres[bs]={'thr':bs/avg,'lat_ms':avg*1000/bs}
    print(f'  Batch={bs:4d}: {bs/avg:7.0f} samples/s  {avg*1000/bs:.2f}ms/sample')

n_params=eval_m.count_params()
print(f'Params:{n_params:,}  FP16~{n_params*2/1e6:.1f}MB  INT8~{n_params/1e6:.1f}MB')
time_check('After FP16 latency/throughput profiling')

# ── Actual TFLite INT8 quantization (real conversion, real measured numbers) ──
# Bounded cost: conversion is one-time (~10-30s, CPU). Accuracy is evaluated
# on a stratified random subsample (600 of ~4800 test signals, 100/SNR) to
# keep wall-clock small while remaining statistically meaningful for a
# 4-class problem. Latency uses the same 200-iteration protocol as FP16.
print('\nRunning TFLite INT8 post-training quantization...')
t_quant0 = time.time()

def rep_data():
    for i in range(200):
        yield [Xe_te[i:i+1], Xg_te[i:i+1], Xs_te[i:i+1]]

converter = tf.lite.TFLiteConverter.from_keras_model(eval_m)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_data
# Allow float fallback ops (gate softmax etc. may not have INT8 kernels);
# this still quantizes the bulk of conv/dense weights to INT8.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS
]
tflite_int8 = converter.convert()
quant_time = time.time() - t_quant0

tflite_path = f'{OUTPUT_DIR}/model_int8.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_int8)
tflite_size_mb = len(tflite_int8) / 1e6
print(f'  Converted in {quant_time:.1f}s -> {tflite_path}  ({tflite_size_mb:.2f} MB)')

# ── Stratified subsample for accuracy + latency evaluation ───────────────────
rng_q = np.random.RandomState(2024)
sub_idx = []
for snr in SNRs:
    idx_snr = np.where(SNR_te == snr)[0]
    sub_idx.extend(rng_q.choice(idx_snr, size=min(100, len(idx_snr)), replace=False))
sub_idx = np.array(sub_idx)
print(f'  Evaluating on {len(sub_idx)} stratified samples ({len(sub_idx)//len(SNRs)}/SNR)')

interpreter = tf.lite.Interpreter(model_content=tflite_int8)
interpreter.allocate_tensors()
inp_details  = interpreter.get_input_details()
out_details  = interpreter.get_output_details()
name_to_idx  = {d['name'].split(';')[0].split(':')[0]: d['index'] for d in inp_details}

def find_input_idx(details, key):
    for d in details:
        if key in d['name']:
            return d['index']
    return details[0]['index']  # fallback

idx_emd = find_input_idx(inp_details, 'emd')
idx_gaf = find_input_idx(inp_details, 'gaf')
idx_seq = find_input_idx(inp_details, 'seq')
out_idx = out_details[0]['index']

int8_preds = np.zeros(len(sub_idx), dtype=np.int64)
int8_times = []
for n, j in enumerate(tqdm(sub_idx, desc='  TFLite INT8 eval', leave=False)):
    interpreter.set_tensor(idx_emd, Xe_te[j:j+1])
    interpreter.set_tensor(idx_gaf, Xg_te[j:j+1])
    interpreter.set_tensor(idx_seq, Xs_te[j:j+1])
    t0 = time.perf_counter()
    interpreter.invoke()
    int8_times.append((time.perf_counter()-t0)*1000)
    out = interpreter.get_tensor(out_idx)
    int8_preds[n] = np.argmax(out[0])

int8_true = Y_te[sub_idx]
int8_snr  = SNR_te[sub_idx]
int8_acc_per_snr = {s: float(np.mean(int8_preds[int8_snr==s]==int8_true[int8_snr==s]))*100 for s in SNRs}
int8_acc_overall = float(np.mean(int8_preds==int8_true))*100

# Compare to FP16/FP32 accuracy on the SAME subsample (fair apples-to-apples)
fp_inp = {'emd_input':Xe_te[sub_idx],'gaf_input':Xg_te[sub_idx],'seq_input':Xs_te[sub_idx]}
fp_preds = eval_m.predict(fp_inp, batch_size=256, verbose=0).argmax(1)
fp_acc_overall = float(np.mean(fp_preds==int8_true))*100
fp_acc_per_snr = {s: float(np.mean(fp_preds[int8_snr==s]==int8_true[int8_snr==s]))*100 for s in SNRs}

int8_lat_mean = np.mean(int8_times[20:]) if len(int8_times)>20 else np.mean(int8_times)
int8_lat_std  = np.std(int8_times[20:])  if len(int8_times)>20 else np.std(int8_times)

print(f'\n  FP16 (GPU) accuracy on subsample : {fp_acc_overall:.2f}%')
print(f'  INT8 (CPU) accuracy on subsample : {int8_acc_overall:.2f}%')
print(f'  Accuracy delta (INT8 - FP16)     : {int8_acc_overall-fp_acc_overall:+.2f} pp')
print(f'  INT8 CPU latency: mean={int8_lat_mean:.2f}ms  std={int8_lat_std:.2f}ms')
print(f'  Model size: FP32 {n_params*4/1e6:.2f}MB -> TFLite INT8 {tflite_size_mb:.2f}MB '
      f'({n_params*4/1e6/tflite_size_mb:.1f}x smaller)')

print('\n  Per-SNR accuracy (FP16 GPU vs INT8 CPU):')
print(f'  {"SNR":>6}{"FP16":>8}{"INT8":>8}{"Delta":>8}')
for s in SNRs:
    d = int8_acc_per_snr[s]-fp_acc_per_snr[s]
    print(f'  {s:>6}{fp_acc_per_snr[s]:8.1f}{int8_acc_per_snr[s]:8.1f}{d:+8.2f}')

# ── Figure (FP16 latency hist/throughput + INT8 comparison) ──────────────────
fig,axes=plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Exp E: Inference Time Profiling + INT8 Quantization (R4.7)')

axes[0].hist(stimes[50:],bins=30,color='steelblue',edgecolor='white',alpha=0.85)
axes[0].axvline(sm,color='red',ls='--',label=f'Mean={sm:.1f}ms')
axes[0].axvline(sp95,color='orange',ls=':',label=f'p95={sp95:.1f}ms')
axes[0].set_xlabel('Latency (ms)'); axes[0].set_ylabel('Count'); axes[0].legend(fontsize=8)
axes[0].set_title('FP16 GPU Single-sample Latency')

bsizes=sorted(bres.keys()); thru=[bres[b]['thr'] for b in bsizes]
axes[1].plot(bsizes,thru,'o-',color='steelblue',lw=2,ms=7)
[axes[1].annotate(f'{t:.0f}',(b,t),textcoords='offset points',xytext=(0,8),ha='center',fontsize=9) for b,t in zip(bsizes,thru)]
axes[1].set_xlabel('Batch size'); axes[1].set_ylabel('Throughput (samples/s)')
axes[1].set_title('FP16 GPU Batch Throughput'); axes[1].grid(alpha=0.3)

axes[2].plot(SNRs,[fp_acc_per_snr[s] for s in SNRs],'o-',color='steelblue',label='FP16 (GPU)',lw=2)
axes[2].plot(SNRs,[int8_acc_per_snr[s] for s in SNRs],'s--',color='coral',label='INT8 (CPU)',lw=2)
axes[2].set_xlabel('SNR (dB)'); axes[2].set_ylabel('Accuracy (%)'); axes[2].set_ylim(0,105)
axes[2].set_title('FP16 vs INT8 Accuracy'); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/expE_inference.png'); plt.show()
print('Saved expE_inference.png')
time_check('After Exp E complete (incl. INT8 quantization)')


In [ ]:
# ── Summary + save (Exp B + D results, separate from main NB5 export) ──────
print('\n' + '='*70)
print('NB5b COMPLETE — Exp B + D + E results')
print('='*70)

print('\nExp B: Sensitivity ranges')
for sweep,params in [('gasf_size',gasf_sizes),('n_imfs',imf_counts),('bigru',bigru_opts)]:
    vs=[np.mean(list(expB[sweep][p].values())) for p in params]
    print(f'  {sweep:<15} best={params[vs.index(max(vs))]}  peak={max(vs):.2f}%  range={max(vs)-min(vs):.2f}pp')

print('\nExp D: Multi-scale GASF')
for name,r in expD.items():
    print(f'  {name:<36} mean={np.mean(list(r.values())):.2f}%  -10dB={r[-10]:.1f}%')

print('\nExp E: Inference + Quantization')
print(f'  FP16 GPU latency: {sm:.1f}+/-{ss:.1f}ms (p95={sp95:.1f}ms)')
print(f'  INT8 CPU latency: {int8_lat_mean:.2f}+/-{int8_lat_std:.2f}ms')
print(f'  FP16 accuracy (subsample): {fp_acc_overall:.2f}%')
print(f'  INT8 accuracy (subsample): {int8_acc_overall:.2f}%  (delta: {int8_acc_overall-fp_acc_overall:+.2f}pp)')
print(f'  Model size: FP32 {n_params*4/1e6:.2f}MB -> INT8 TFLite {tflite_size_mb:.2f}MB')

total_h = (time.time() - _t_start) / 3600
print(f'\nTotal wall-clock time: {total_h:.2f}h (budget: {TIME_BUDGET_HOURS}h)')
print('Note: This run used the lighter Exp B/D protocol (reduced dataset size,')
print('reduced epochs/patience) — see header for details. These results answer')
print('the RELATIVE sensitivity question for the reviewer response; if the')
print('paper needs absolute headline accuracy for a specific config, retrain')
print('that one configuration with the main NB5 full protocol.')

nb5b_out = {'expB':expB, 'expD':expD,
            'expE_times':stimes, 'expE_batch':bres,
            'expE_int8_acc_per_snr':int8_acc_per_snr,
            'expE_int8_acc_overall':int8_acc_overall,
            'expE_fp_acc_per_snr':fp_acc_per_snr,
            'expE_fp_acc_overall':fp_acc_overall,
            'expE_int8_latency_mean':int8_lat_mean,
            'expE_int8_latency_std':int8_lat_std,
            'expE_tflite_size_mb':tflite_size_mb,
            'protocol': {'N_PER_CLASS':N_PER_CLASS,'N_TEST':N_TEST,
                        'EPOCHS':EPOCHS,'PATIENCE':PATIENCE},
            'wall_clock_hours': total_h}
pkl_path = f'{OUTPUT_DIR}/nb5b_export.pkl'
with open(pkl_path,'wb') as f: pickle.dump(nb5b_out,f)
print(f'\nSaved {pkl_path}')

zip_path = f'{OUTPUT_DIR}/nb5b_figures.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['expB_sensitivity.png','expD_multiscale.png','expE_inference.png']:
        fp = f'{OUTPUT_DIR}/{fn}'
        if os.path.exists(fp): zf.write(fp, fn)
    tflite_fp = f'{OUTPUT_DIR}/model_int8.tflite'
    if os.path.exists(tflite_fp): zf.write(tflite_fp, 'model_int8.tflite')
print(f'Figures zipped to {zip_path}')
print('\nNB5b complete.')
